<a href="https://colab.research.google.com/github/malith2003k/Statistical-Learning-e22039/blob/main/data%20wrandling%20assignment%20e22039.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import io
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats
from google.colab import files
from IPython.display import HTML, display

class PlottingMethods:

    @staticmethod
    def _wrap_html(fig) -> dict:
        try:
            html_str = fig.to_html(include_plotlyjs='cdn', full_html=False)
            return {"status": "success", "html": html_str}
        except Exception as e:
            return {"status": "error", "message": str(e)}

    def display_image(self, result: dict):
        if result.get("status") == "success" and "html" in result:
            display(HTML(result["html"]))
        else:
            print(f"Error displaying chart: {result.get('message', 'Unknown error')}")

    def plot_bar_chart(self, x: str, y: str, data: pd.DataFrame, color: str = None, barmode: str = 'group', title: str = None) -> dict:
        if data is None or data.empty:
            return {"status": "error", "message": "Provided DataFrame is empty or None."}
        fig = px.bar(data, x=x, y=y, color=color, barmode=barmode, title=title or f"{y} by {x}")
        return self._wrap_html(fig)

    def plot_pie_chart(self, names: str, values: str, data: pd.DataFrame, hole: float = 0.4, title: str = None) -> dict:
        if data is None or data.empty:
            return {"status": "error", "message": "Provided DataFrame is empty or None."}
        fig = px.pie(data, names=names, values=values, hole=hole, title=title or f"Distribution of {names}")
        return self._wrap_html(fig)

    def plot_histogram(self, x: str, data: pd.DataFrame, bins: list = None, title: str = None) -> dict:
        if data is None or data.empty:
            return {"status": "error", "message": "Provided DataFrame is empty or None."}

        fig = px.histogram(data, x=x, title=title or f"Histogram of {x}")
        if bins:
            fig.update_traces(xbins=dict(start=bins[0], end=bins[-1], size=(bins[1]-bins[0] if len(bins)>1 else None)))
        return self._wrap_html(fig)

    def plot_heat_map(self, values: str, index: str, columns: str, data: pd.DataFrame, aggregate_method: str = 'mean', title: str = None) -> dict:
        if data is None or data.empty:
            return {"status": "error", "message": "Provided DataFrame is empty or None."}
        pivot_df = data.pivot_table(values=values, index=index, columns=columns, aggfunc=aggregate_method)
        fig = px.imshow(pivot_df, labels=dict(x=columns, y=index, color=values), title=title or "Heatmap Matrix")
        return self._wrap_html(fig)


class DataInspector:
    def __init__(self):
        self.df = None
        self.plotter = PlottingMethods()

    def upload_data(self):
        print("Please select your CSV file to upload:")
        uploaded = files.upload()
        if not uploaded:
            print("Upload cancelled.")
            return

        file_name = list(uploaded.keys())[0]
        garbage_values = ['?', 'n/a', 'N/A', 'NULL', 'null', ' ', '']
        self.df = pd.read_csv(io.BytesIO(uploaded[file_name]), na_values=garbage_values)
        print(f"Successfully loaded '{file_name}' with {self.df.shape[0]} rows and {self.df.shape[1]} columns.")
        self._auto_type_correction()

    def _auto_type_correction(self):
        if self.df is None: return
        for col in self.df.columns:
            if self.df[col].dtype == 'object':
                converted = pd.to_numeric(self.df[col], errors='coerce')
                if not converted.isna().all():
                    self.df[col] = converted

    def get_summary(self):
        if self.df is None:
            print("No active DataFrame found. Please call upload_data() first.")
            return

        print("=== STRUCTURAL DATA SUMMARY ===")
        print(f"Total Rows: {self.df.shape[0]} | Total Columns: {self.df.shape[1]}\n")

        numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()

        print(f"Numerical Columns ({len(numeric_cols)}): {numeric_cols}")
        print(f"Categorical Columns ({len(categorical_cols)}): {categorical_cols}\n")
        print("=== FIRST 20 ROWS PREVIEW ===")
        display(self.df.head(20))

    def handle_missing_values(self, strategy: str = 'median', fill_value=None):
        if self.df is None: return

        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        categorical_cols = self.df.select_dtypes(exclude=[np.number]).columns

        for col in self.df.columns:
            if self.df[col].isna().sum() == 0:
                continue

            if strategy == 'constant':
                self.df[col] = self.df[col].fillna(fill_value)
            elif col in numeric_cols:
                if strategy == 'mean':
                    self.df[col] = self.df[col].fillna(self.df[col].mean())
                elif strategy == 'median':
                    self.df[col] = self.df[col].fillna(self.df[col].median())
                elif strategy == 'mode':
                    self.df[col] = self.df[col].fillna(self.df[col].mode()[0])
            elif col in categorical_cols:
                self.df[col] = self.df[col].fillna(self.df[col].mode()[0])
        print(f"Missing values handled successfully using strategy: '{strategy}'.")

    def remove_duplicates(self):
        if self.df is None: return
        initial_count = len(self.df)
        self.df.drop_duplicates(inplace=True)
        print(f"Removed {initial_count - len(self.df)} exact duplicate rows.")

    def handle_outliers(self, columns: list, find_and_delete: bool = False):
        if self.df is None: return
        mask = pd.Series(True, index=self.df.index)

        for col in columns:
            if col in self.df.columns and pd.api.types.is_numeric_dtype(self.df[col]):
                Q1 = self.df[col].quantile(0.25)
                Q3 = self.df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR

                col_mask = (self.df[col] >= lower_bound) & (self.df[col] <= upper_bound)
                outliers_count = len(self.df) - col_mask.sum()
                print(f"Column '{col}': Identified {outliers_count} outliers outside range [{lower_bound:.2f}, {upper_bound:.2f}]")
                mask = mask & col_mask

        if find_and_delete:
            initial_count = len(self.df)
            self.df = self.df[mask]
            print(f"Pruned out {initial_count - len(self.df)} rows containing anomalies.")

    def delete_columns(self, input_str: str = None):
        if self.df is None: return
        if not input_str:
            input_str = input("Enter column names to delete (comma-separated): ")
        cols_to_delete = [c.strip() for c in input_str.split(',') if c.strip() in self.df.columns]
        self.df.drop(columns=cols_to_delete, inplace=True)
        print(f"Deleted columns: {cols_to_delete}")

    def delete_rows(self, input_str: str = None):
        if self.df is None: return
        if not input_str:
            input_str = input("Enter row index positions to delete (comma-separated): ")
        try:
            indices_to_delete = [int(i.strip()) for i in input_str.split(',') if i.strip().isdigit()]
            valid_indices = [i for i in indices_to_delete if i in self.df.index]
            self.df.drop(index=valid_indices, inplace=True)
            print(f"Deleted rows at indices: {valid_indices}")
        except Exception as e:
            print(f"Error executing targeted row deletion: {e}")

    def extract_normalized_numeric_data(self, method: str = 'standard') -> pd.DataFrame:
        numeric_df = self.df.select_dtypes(include=[np.number]).copy()
        if numeric_df.empty: return numeric_df

        for col in numeric_df.columns:
            if method == 'minmax':
                min_val, max_val = numeric_df[col].min(), numeric_df[col].max()
                if max_val != min_val:
                    numeric_df[col] = (numeric_df[col] - min_val) / (max_val - min_val)
            elif method == 'standard':
                mean, std = numeric_df[col].mean(), numeric_df[col].std()
                if std != 0:
                    numeric_df[col] = (numeric_df[col] - mean) / std
            elif method == 'robust':
                q25, median, q75 = numeric_df[col].quantile(0.25), numeric_df[col].median(), numeric_df[col].quantile(0.75)
                iqr = q75 - q25
                if iqr != 0:
                    numeric_df[col] = (numeric_df[col] - median) / iqr
        return numeric_df

    def extract_normalized_categorical_data(self, method: str = 'onehot') -> pd.DataFrame:
        cat_df = self.df.select_dtypes(exclude=[np.number]).copy()
        if cat_df.empty: return cat_df

        if method == 'onehot':
            return pd.get_dummies(cat_df, drop_first=False, dtype=float)
        elif method == 'ordinal':
            for col in cat_df.columns:
                cat_df[col] = cat_df[col].astype('category').cat.codes.astype(float)
            return cat_df
        elif method == 'uniform':
            for col in cat_df.columns:
                codes = cat_df[col].astype('category').cat.codes
                if codes.max() != 0:
                    cat_df[col] = (codes / codes.max()).astype(float)
                else:
                    cat_df[col] = codes.astype(float)
            return cat_df
        return cat_df

    def create_normalized_data_df(self, num_method: str = 'standard', cat_method: str = 'onehot') -> pd.DataFrame:
        num_part = self.extract_normalized_numeric_data(method=num_method)
        cat_part = self.extract_normalized_categorical_data(method=cat_method)
        return pd.concat([num_part, cat_part], axis=1)

    def plot_numerical(self, column_names: list):
        if self.df is None: return

        for col in column_names:
            if col not in self.df.columns or not pd.api.types.is_numeric_dtype(self.df[col]):
                continue

            fig = make_subplots(
                rows=1, cols=3,
                subplot_titles=(f"Violin/Box Summary", f"Index Plot Sequence", f"Frequency Density")
            )

            fig.add_trace(go.Violin(x=self.df[col], box_visible=True, meanline_visible=True, name=col, showlegend=False), row=1, col=1)
            fig.add_trace(go.Scatter(y=self.df[col], mode='markers', marker=dict(opacity=0.6), showlegend=False), row=1, col=2)
            fig.add_trace(go.Histogram(x=self.df[col], showlegend=False), row=1, col=3)

            fig.update_layout(title_text=f"Continuous Diagnostic Analysis Matrix: {col}", height=400, width=1050)
            fig.show()

    def plot_categorical(self, column_names: list):
        if self.df is None: return
        for col in column_names:
            if col not in self.df.columns: continue

            counts = self.df[col].value_counts()
            percentages = self.df[col].value_counts(normalize=True) * 100

            text_labels = [f"{c} ({p:.1f}%)" for c, p in zip(counts, percentages)]

            fig = go.Figure(data=[go.Bar(x=counts.index, y=counts.values, text=text_labels, textposition='auto')])
            fig.update_layout(title=f"Categorical Frequency Distribution: {col}", yaxis_title="Counts", height=450)
            fig.show()

    def plot_relationship(self, target_a: str, target_b: str):
        if self.df is None or target_a not in self.df.columns or target_b not in self.df.columns: return

        is_a_num = pd.api.types.is_numeric_dtype(self.df[target_a])
        is_b_num = pd.api.types.is_numeric_dtype(self.df[target_b])

        if is_a_num and is_b_num:
            fig = px.scatter(self.df, x=target_a, y=target_b, trendline="ols", title=f"Bivariate Scatter Evaluation: {target_a} vs {target_b}")
            fig.show()
        elif (is_a_num and not is_b_num) or (not is_a_num and is_b_num):
            num_col = target_a if is_a_num else target_b
            cat_col = target_b if is_a_num else target_a
            fig = px.box(self.df, x=cat_col, y=num_col, points="all", title=f"Distribution Profile: {num_col} stratified by {cat_col}")
            fig.show()
        else:
            pivot_data = self.df.groupby([target_a, target_b]).size().reset_index(name='Count')
            fig = px.bar(pivot_data, x=target_a, y='Count', color=target_b, barmode='group', title=f"Cross-Tabulation Matrix Count: {target_a} vs {target_b}")
            fig.show()

    def _cramers_v(self, x, y):
        confusion_matrix = pd.crosstab(x, y)
        chi2 = stats.chi2_contingency(confusion_matrix)[0]
        n = confusion_matrix.sum().sum()
        phi2 = chi2 / n
        r, k = confusion_matrix.shape
        phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
        rcorr = r - ((r-1)**2)/(n-1)
        kcorr = k - ((k-1)**2)/(n-1)
        return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1))) if min((kcorr-1), (rcorr-1)) > 0 else 0.0

    def plot_all_associations_heatmap(self):
        if self.df is None: return

        cols = self.df.columns.tolist()
        n_cols = len(cols)
        assoc_matrix = pd.DataFrame(np.zeros((n_cols, n_cols)), index=cols, columns=cols)

        for i in range(n_cols):
            for j in range(n_cols):
                col_i = cols[i]
                col_j = cols[j]

                if i == j:
                    assoc_matrix.iloc[i, j] = 1.0
                    continue

                is_i_num = pd.api.types.is_numeric_dtype(self.df[col_i])
                is_j_num = pd.api.types.is_numeric_dtype(self.df[col_j])

                valid_idx = self.df[[col_i, col_j]].dropna().index
                if len(valid_idx) < 2:
                    assoc_matrix.iloc[i, j] = np.nan
                    continue

                x_val = self.df.loc[valid_idx, col_i]
                y_val = self.df.loc[valid_idx, col_j]

                if is_i_num and is_j_num:
                    r_val, _ = stats.pearsonr(x_val, y_val)
                    assoc_matrix.iloc[i, j] = r_val
                elif not is_i_num and not is_j_num:
                    assoc_matrix.iloc[i, j] = self._cramers_v(x_val, y_val)
                else:
                    num_v = x_val if is_i_num else y_val
                    cat_v = y_val if is_i_num else x_val

                    categories = cat_v.unique()
                    if len(categories) == 2:
                        try:
                            cat_codes = cat_v.astype('category').cat.codes
                            pb_r, _ = stats.pointbiserialr(cat_codes, num_v)
                            assoc_matrix.iloc[i, j] = pb_r
                        except:
                            assoc_matrix.iloc[i, j] = 0.0
                    else:
                        try:
                            groups = [num_v[cat_v == cat].values for cat in categories]
                            f_val, _ = stats.f_oneway(*groups)
                            k_groups = len(categories)
                            n_total = len(num_v)
                            eta2 = (f_val * (k_groups - 1)) / (f_val * (k_groups - 1) + (n_total - k_groups))
                            assoc_matrix.iloc[i, j] = np.sqrt(eta2) if not np.isnan(eta2) else 0.0
                        except:
                            assoc_matrix.iloc[i, j] = 0.0

        fig = px.imshow(
            assoc_matrix,
            text_auto=".2f",
            color_continuous_scale='RdBu_r',
            zmin=-1.0, zmax=1.0,
            title="Unified Association Heatmap Matrix (Pearson / Cramér's V / Eta)"
        )
        fig.show()


inspector = DataInspector()
titanic_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
inspector.df = pd.read_csv(titanic_url)
print("Initial Structural State loaded directly for testing.\n")

inspector.handle_missing_values(strategy='median')
inspector.remove_duplicates()
inspector.handle_outliers(columns=['Fare', 'Age'], find_and_delete=True)
inspector.get_summary()

inspector.plot_numerical(['Age', 'Fare'])
inspector.plot_categorical(['Survived', 'Pclass'])
inspector.plot_relationship('Pclass', 'Fare')
inspector.plot_all_associations_heatmap()

model_ready_df = inspector.create_normalized_data_df(num_method='robust', cat_method='onehot')
print("\n=== MACHINE LEARNING ENCODED FRAME PREVIEW ===")
display(model_ready_df.head(5))



Initial Structural State loaded directly for testing.

Missing values handled successfully using strategy: 'median'.
Removed 0 exact duplicate rows.
Column 'Fare': Identified 116 outliers outside range [-26.72, 65.63]
Column 'Age': Identified 66 outliers outside range [2.50, 54.50]
Pruned out 170 rows containing anomalies.
=== STRUCTURAL DATA SUMMARY ===
Total Rows: 721 | Total Columns: 12

Numerical Columns (7): ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Categorical Columns (5): ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']

=== FIRST 20 ROWS PREVIEW ===


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,B96 B98,S
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,B96 B98,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,B96 B98,S
5,6,0,3,"Moran, Mr. James",male,28.0,0,0,330877,8.4583,B96 B98,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,B96 B98,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,B96 B98,C
10,11,1,3,"Sandstrom, Miss. Marguerite Rut",female,4.0,1,1,PP 9549,16.7000,G6,S
12,13,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.0500,B96 B98,S


/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:573: SmallSampleWarning:

all input arrays have length 1.  f_oneway requires that at least one input has length greater than 1.

/tmp/ipykernel_4883/2157357218.py:325: SmallSampleWarning:

One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:573: SmallSampleWarning:

all input arrays have length 1.  f_oneway requires that at least one input has length greater than 1.

/tmp/ipykernel_4883/2157357218.py:325: SmallSampleWarning:

One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:573: SmallSampleWarning:

all input arrays have length 1.  f_oneway requires that at least one input has length greater than 1.

/tmp/ipykernel_4883/2157357


=== MACHINE LEARNING ENCODED FRAME PREVIEW ===


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,"Name_Abbing, Mr. Anthony","Name_Abbott, Mr. Rossmore Edward","Name_Abbott, Mrs. Stanton (Rosa Hunt)",...,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Embarked_C,Embarked_Q,Embarked_S
0,-0.984581,0.0,0.0,-0.545455,1.0,0,-0.277560,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.980176,1.0,0.0,-0.181818,0.0,0,-0.240276,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.977974,1.0,-2.0,0.636364,1.0,0,2.255002,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.975771,0.0,0.0,0.636364,0.0,0,-0.233371,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
5,-0.973568,0.0,0.0,0.000000,0.0,0,-0.210818,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
